Upload the ZIP file

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving archive (11).zip to archive (11).zip


In [ ]:
import zipfile
import io
import os

csv_files = []

for fn in uploaded.keys():
    if fn.endswith(".zip"):
        print("Extracting:", fn)
        with zipfile.ZipFile(io.BytesIO(uploaded[fn]), 'r') as zip_ref:
            zip_ref.extractall()

# Detect CSV files
csv_files = [f for f in os.listdir() if f.endswith(".csv")]

if not csv_files:
    raise Exception("No CSV files found! Please upload a ZIP containing .csv files.")

print("Detected CSV files:", csv_files)

DATA_PATH = csv_files[0]
print("Using CSV:", DATA_PATH)


Extracting: archive (11).zip
Detected CSV files: ['INFY.csv', 'NIFTY50_all.csv', 'ONGC.csv', 'TATAMOTORS.csv', 'TCS.csv', 'HINDUNILVR.csv', 'INDUSINDBK.csv', 'HDFCBANK.csv', 'GAIL.csv', 'LT.csv', 'SHREECEM.csv', 'HEROMOTOCO.csv', 'GRASIM.csv', 'AXISBANK.csv', 'SUNPHARMA.csv', 'HDFC.csv', 'HINDALCO.csv', 'ITC.csv', 'BAJFINANCE.csv', 'ICICIBANK.csv', 'POWERGRID.csv', 'BAJAJFINSV.csv', 'MARUTI.csv', 'BHARTIARTL.csv', 'EICHERMOT.csv', 'IOC.csv', 'SBIN.csv', 'TATASTEEL.csv', 'BAJAJ-AUTO.csv', 'NESTLEIND.csv', 'ZEEL.csv', 'WIPRO.csv', 'RELIANCE.csv', 'stock_metadata.csv', 'JSWSTEEL.csv', 'COALINDIA.csv', 'BPCL.csv', 'HCLTECH.csv', 'INFRATEL.csv', 'TITAN.csv', 'TECHM.csv', 'CIPLA.csv', 'UPL.csv', 'DRREDDY.csv', 'MM.csv', 'VEDL.csv', 'KOTAKBANK.csv', 'NTPC.csv', 'BRITANNIA.csv', 'ULTRACEMCO.csv', 'ASIANPAINT.csv', 'ADANIPORTS.csv']
📌 Using CSV: INFY.csv


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv(DATA_PATH)

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

df.head()


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble
0,2000-01-03,INFOSYSTCH,EQ,14467.75,15625.00,15625.20,15625.00,15625.20,15625.20,15625.18,5137,8.026657e+12,NaN,NaN,NaN
1,2000-01-04,INFOSYSTCH,EQ,15625.20,16800.00,16875.25,16253.00,16875.25,16855.90,16646.38,56186,9.352937e+13,NaN,NaN,NaN
2,2000-01-05,INFOSYSTCH,EQ,16855.90,15701.00,16250.00,15507.45,15507.45,15507.45,15786.38,164605,2.598516e+14,NaN,NaN,NaN
3,2000-01-06,INFOSYSTCH,EQ,15507.45,15256.65,15300.00,14266.85,14266.85,14266.85,14462.82,81997,1.185908e+14,NaN,NaN,NaN
4,2000-01-07,INFOSYSTCH,EQ,14266.85,13125.50,13125.50,13125.50,13125.50,13125.50,13125.50,7589,9.960942e+12,NaN,NaN,NaN


In [ ]:
df["return_1d"] = df["Close"].pct_change()
df["log_return_1d"] = np.log(df["Close"] / df["Close"].shift(1))

for w in [5, 10, 20, 50]:
    df[f"ma_{w}"] = df["Close"].rolling(w).mean()

for w in [5, 10, 20]:
    df[f"vol_{w}"] = df["Close"].pct_change().rolling(w).std()

for lag in [1, 2, 3, 5]:
    df[f"close_lag_{lag}"] = df["Close"].shift(lag)
    df[f"vol_lag_{lag}"] = df["Volume"].shift(lag)

df["close_next"] = df["Close"].shift(-1)
df["up_next"] = (df["close_next"] > df["Close"]).astype(int)

df = df.dropna()
df.head()


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,...,close_lag_1,vol_lag_1,close_lag_2,vol_lag_2,close_lag_3,vol_lag_3,close_lag_5,vol_lag_5,close_next,up_next
2850,2011-06-01,INFOSYSTCH,EQ,2785.65,2794.00,2824.00,2784.00,2811.95,2811.55,2804.92,...,2785.65,2551326.0,2780.60,858419.0,2787.50,783118.0,2788.65,1384427.0,2803.05,0
2851,2011-06-02,INFOSYSTCH,EQ,2811.55,2780.00,2820.00,2777.00,2800.00,2803.05,2800.74,...,2811.55,1013233.0,2785.65,2551326.0,2780.60,858419.0,2776.55,1059522.0,2816.20,1
2852,2011-06-03,INFOSYSTCH,EQ,2803.05,2811.00,2842.80,2810.90,2818.70,2816.20,2825.76,...,2803.05,889661.0,2811.55,1013233.0,2785.65,2551326.0,2787.50,783118.0,2837.90,1
2853,2011-06-06,INFOSYSTCH,EQ,2816.20,2822.00,2845.95,2795.15,2845.00,2837.90,2822.59,...,2816.20,786673.0,2803.05,889661.0,2811.55,1013233.0,2780.60,858419.0,2892.55,1
2854,2011-06-07,INFOSYSTCH,EQ,2837.90,2830.05,2904.00,2830.05,2889.60,2892.55,2878.65,...,2837.90,471158.0,2816.20,786673.0,2803.05,889661.0,2785.65,2551326.0,2871.70,0


In [ ]:
features = df.select_dtypes(include=[np.number]).columns.tolist()
features = [f for f in features if f not in ["close_next", "up_next", "Close"]]

X = df[features].values
y_clf = df["up_next"].values
y_reg = df["close_next"].values


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clf, test_size=0.2, shuffle=False
)


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=10)
X_train_p = pca.fit_transform(X_train_s)
X_test_p = pca.transform(X_test_s)

pca.explained_variance_ratio_


array([0.56420608, 0.13073827, 0.08520666, 0.05484094, 0.04077442,
       0.03570768, 0.02194206, 0.01788071, 0.01578737, 0.01206323])

In [ ]:
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_p, y_train)

svm = SVC(kernel="rbf", C=10, gamma='scale')
svm.fit(X_train_bal, y_train_bal)

y_pred = svm.predict(X_test_p)

print("SVM Classification Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



=== 📈 SVM Classification Results ===
Accuracy: 0.48879837067209775
              precision    recall  f1-score   support

           0       0.44      0.49      0.46       222
           1       0.54      0.49      0.51       269

    accuracy                           0.49       491
   macro avg       0.49      0.49      0.49       491
weighted avg       0.49      0.49      0.49       491

Confusion Matrix:
 [[109 113]
 [138 131]]


In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, shuffle=False
)

scaler_r = StandardScaler()
X_train_rs = scaler_r.fit_transform(X_train_r)
X_test_rs = scaler_r.transform(X_test_r)

lr = LinearRegression()
lr.fit(X_train_rs, y_train_r)

y_pred_lr = lr.predict(X_test_rs)

print("Linear Regression Results")
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_lr)))
print("MAE:", mean_absolute_error(y_test_r, y_pred_lr))
print("R2:", r2_score(y_test_r, y_pred_lr))



=== 📉 Linear Regression Results ===
RMSE: 19.686191330437406
MAE: 13.820706565232934
R2: 0.9932669486728767
